# 1) Bronze Layer
## Barcelona Urban Intelligence Assistant
### Advanced Data Processing and Analysis


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import json
import os
import glob

# Working directory
os.chdir(r"C:\Users\zarah\Downloads\Spring2026\Advanced Data\Final Project")
print("Working directory:", os.getcwd())

# Connect to DuckDB
con = duckdb.connect("barcelona_urban.db")
con.execute("CREATE SCHEMA IF NOT EXISTS bronze")
con.execute("CREATE SCHEMA IF NOT EXISTS silver")
con.execute("CREATE SCHEMA IF NOT EXISTS gold")
print("Connected to barcelona_urban.db — schemas ready")
print("DuckDB version:", duckdb.__version__)


Working directory: C:\Users\zarah\Downloads\Spring2026\Advanced Data\Final Project
Connected to barcelona_urban.db — schemas ready
DuckDB version: 1.4.4


## 1. IRIS Citizen Complaints 2025

The IRIS system is Barcelona City Council's official channel for citizens to report urban problems (for example broken street lights and noise complaints), every complaint is date-stamped, district-coded, categorised, and tracked until the resolution is found.

This is our primary dataset for research question 3 (what do complaints reveal about urban service quality?), this dataset has over 200,000+ rows and can gives us statistically meaningful comparisons between all 10 districts.

For the bronze layer the dataset is loaded with `dtype=str` for all columns, since Barcelona's export file contains dates stored as separate integer columns (day, month, year). Complaint IDs that look like numbers have to be treated as identifiers, and district codes that can have leading zeros in them.

In [ ]:
# IRIS Citizen Complaints

iris_raw = pd.read_csv(
    "2025_IRIS_Peticions_Ciutadanes_OpenData.csv",
    dtype=str,
    encoding="utf-8",
    on_bad_lines="skip"
)

iris_raw.columns = [c.strip('"').strip() for c in iris_raw.columns]

for col in iris_raw.columns:
    iris_raw[col] = iris_raw[col].astype(str).str.strip('"').str.strip()

iris_raw.replace("nan", np.nan, inplace=True)

print(f"IRIS raw shape: {iris_raw.shape}")
print("Columns:", list(iris_raw.columns))
iris_raw.head(3)


IRIS raw shape: (287304, 25)
Columns: ['FITXA_ID', 'TIPUS', 'AREA', 'ELEMENT', 'DETALL', 'DIA_DATA_ALTA', 'MES_DATA_ALTA', 'ANY_DATA_ALTA', 'DIA_DATA_TANCAMENT', 'MES_DATA_TANCAMENT', 'ANY_DATA_TANCAMENT', 'CODI_DISTRICTE', 'DISTRICTE', 'CODI_BARRI', 'BARRI', 'SECCIO_CENSAL', 'TIPUS_VIA', 'CARRER', 'NUMERO', 'COORDENADA_X', 'COORDENADA_Y', 'LONGITUD', 'LATITUD', 'SUPORT', 'CANALS_RESPOSTA']


,FITXA_ID,TIPUS,AREA,ELEMENT,DETALL,DIA_DATA_ALTA,MES_DATA_ALTA,ANY_DATA_ALTA,DIA_DATA_TANCAMENT,MES_DATA_TANCAMENT,...,SECCIO_CENSAL,TIPUS_VIA,CARRER,NUMERO,COORDENADA_X,COORDENADA_Y,LONGITUD,LATITUD,SUPORT,CANALS_RESPOSTA
0,38755908,INCIDENCIA,Mobilitat,Vehicles a motor,Vehicles motor abandonats,25,11,2024,01,01,...,NaN,Carrer,de Biscaia,0383,431890,4585512,NaN,NaN,WEB,IMMEDIATA
1,38755913,QUEIXA,Mobilitat,Vehicles a motor,Vehicles motor disciplina viària,28,11,2024,01,01,...,NaN,Carrer,Gran de Sant Andreu,0337,432248,4587660,NaN,NaN,WEB,EMAIL
2,38755907,QUEIXA,Prevenció i seguretat,Molèsties espai públic,Persones col·lectius molestant espai públic,29,11,2024,01,01,...,034,Carrer,de Bartrina,0013,432076,4588026,NaN,NaN,TELÈFON,EMAIL


In [ ]:
# Load into DuckDB bronze
con.execute("CREATE OR REPLACE TABLE bronze.iris_raw AS SELECT * FROM iris_raw")
count = con.execute("SELECT COUNT(*) FROM bronze.iris_raw").fetchone()[0]
print(f"bronze.iris_raw: {count:,} rows loaded")
print("Any change to the source requires re-running this cell from the original CSV.")


bronze.iris_raw: 287,304 rows loaded
Any change to the source requires re-running this cell from the original CSV.


## 2. Traffic Accidents 2025

In [ ]:
# ── Accidents ──────────────────────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE bronze.accidents_raw AS
    SELECT * FROM read_csv_auto(
        '2025_accidents_causa_conductor_gu_bcn.csv',
        header = true,
        ignore_errors = true,
        encoding = 'utf-8'
    )
""")
count = con.execute("SELECT COUNT(*) FROM bronze.accidents_raw").fetchone()[0]
print(f"bronze.accidents_raw: {count:,} rows")
print("Columns:", [r[0] for r in con.execute("DESCRIBE bronze.accidents_raw").fetchall()])


bronze.accidents_raw: 8,072 rows
Columns: ['Número_expedient', 'Codi_districte', 'Nom_districte', 'Codi_barri', 'Nom_barri', 'Codi_carrer', 'Nom_carrer', 'Num_postal', 'Descripcio_dia_setmana', 'Nk_Any', 'Mes_any', 'Nom_mes', 'Dia_mes', 'Hora_dia', 'Descripcio_torn', 'Causa_conductor', 'Coordenada_UTM_X_ED50', 'Coordenada_UTM_Y_ED50', 'Longitud_WGS84', 'Latitud_WGS84']


## 3. Air Quality Monthly Readings 2025

This dataset shows the ourly pollutant measurements, with each row representing one day at one station (with 24 hourly readings H01–H24 and 24 validation flags V01–V24, where 'V' means valid and 'N' means instrument error or calibration period).

Each column pair (H01, V01) represents one hour


In [ ]:
# Air quality: loading all 11 monthly CSVs into one bronze table
aq_files = sorted(glob.glob("Airquality_2025/*.csv"))
print(f"Found {len(aq_files)} air quality files:")
for f in aq_files:
    print(f"  {os.path.basename(f)}")

aq_frames = []
for f in aq_files:
    df = pd.read_csv(f, dtype=str, encoding="utf-8")
    df.columns = [c.strip('"').strip().upper() for c in df.columns]
    df["SOURCE_FILE"] = os.path.basename(f)
    aq_frames.append(df)

aq_raw = pd.concat(aq_frames, ignore_index=True)
print(f"\nTotal rows across all months: {len(aq_raw):,}")
print("Months loaded:", sorted(aq_raw["MES"].dropna().unique().tolist()))


Found 11 air quality files:
  2025_01_Gener_qualitat_aire_BCN.csv
  2025_02_Febrer_qualitat_aire_BCN.csv
  2025_03_Marc_qualitat_aire_BCN.csv
  2025_04_Abril_qualitat_aire_BCN.csv
  2025_06_Juny_qualitat_aire_BCN.csv
  2025_07_Juliol_qualitat_aire_BCN.csv
  2025_08_Agost_qualitat_aire_BCN.csv
  2025_09_Setembre_qualitat_aire_BCN.csv
  2025_10_Octubre_qualitat_aire_BCN.csv
  2025_11_Novembre_qualitat_aire_BCN.csv
  2025_12_Desembre_qualitat_aire_BCN.csv

Total rows across all months: 22,960
Months loaded: ['1', '10', '11', '12', '2', '3', '4', '6', '7', '8', '9']


In [ ]:
con.execute("CREATE OR REPLACE TABLE bronze.airquality_raw AS SELECT * FROM aq_raw")
count = con.execute("SELECT COUNT(*) FROM bronze.airquality_raw").fetchone()[0]
print(f"bronze.airquality_raw: {count:,} rows")


bronze.airquality_raw: 22,960 rows


## 4. Air Quality Station Metadata
Maps each station (ESTACIO code) to its district and neighborhood.

In [ ]:
# Air quality stations
con.execute("""
    CREATE OR REPLACE TABLE bronze.airquality_stations_raw AS
    SELECT * FROM read_csv_auto(
        '2025_qualitat_aire_estacions.csv',
        header = true,
        ignore_errors = true,
        encoding = 'utf-8'
    )
""")
count = con.execute("SELECT COUNT(*) FROM bronze.airquality_stations_raw").fetchone()[0]
print(f"bronze.airquality_stations_raw: {count:,} rows")
display(con.execute("SELECT * FROM bronze.airquality_stations_raw LIMIT 5").df())


bronze.airquality_stations_raw: 55 rows


,Estacio,nom_cabina,codi_dtes,zqa,codi_eoi,Longitud,Latitud,ubicacio,Codi_districte,Nom_districte,Codi_barri,Nom_barri,Clas_1,Clas_2,Codi_Contaminant
0,4,Barcelona - Poblenou,I2,1,8019004,2.2045,41.4039,Plaça Josep Trueta (Pujades - Lope de Vega),10,Sant Marti,68,el Poblenou,Urbana,Fons,8
1,4,Barcelona - Poblenou,I2,1,8019004,2.2045,41.4039,Plaça Josep Trueta (Pujades - Lope de Vega),10,Sant Marti,68,el Poblenou,Urbana,Fons,10
2,4,Barcelona - Poblenou,I2,1,8019004,2.2045,41.4039,Plaça Josep Trueta (Pujades - Lope de Vega),10,Sant Marti,68,el Poblenou,Urbana,Fons,7
3,4,Barcelona - Poblenou,I2,1,8019004,2.2045,41.4039,Plaça Josep Trueta (Pujades - Lope de Vega),10,Sant Marti,68,el Poblenou,Urbana,Fons,12
4,42,Barcelona - Sants,ID,1,8019042,2.1331,41.3788,Jardins de Can Mantega (Joan Güell - Violant d...,3,Sants-Montjuic,18,Sants,Urbana,Fons,8


## 5. EV Charging Stations: all quarters of 2025
The open data Barcelona website had four quarterly datasets in JSON format, we load all four and combine them into one bronze table.
Each station has the coordinates (lat/lon) but no district column, the districts are assigned in the Silver layer.

In [ ]:
# EV Charging: loading all 4 quarterly JSON files from Charginglocations/ folder
import glob

ev_files = sorted(glob.glob("Charginglocations/*.json"))
print(f"Found {len(ev_files)} EV charging files:")
for f in ev_files:
    print(f"  {os.path.basename(f)}")

ev_records = []

# Extracting the quarter labels from the filenames (ex: 2025_1T_ becomes Q1)
for filepath in ev_files:
    basename = os.path.basename(filepath)
    quarter_raw = basename.split("_")[1]
    quarter = "Q" + quarter_raw[0]

    with open(filepath, "r", encoding="utf-8") as f:
        ev_json = json.load(f)

    locations = ev_json.get("locations", [])
    print(f"  {quarter}: {len(locations)} stations found")

    for loc in locations:
        coords  = loc.get("coordinates", {})
        address = loc.get("address", {})
        ev_records.append({
            "station_id"   : loc.get("id"),
            "quarter"      : quarter,              # Q1 / Q2 / Q3 / Q4
            "network_name" : loc.get("network_brand_name"),
            "latitude"     : coords.get("latitude"),
            "longitude"    : coords.get("longitude"),
            "address_str"  : address.get("address_string"),
            "postal_code"  : address.get("postal_code"),
            "locality"     : address.get("locality"),
            "access"       : loc.get("access_restriction"),
        })

ev_raw = pd.DataFrame(ev_records)
print(f"\nCombined EV raw: {len(ev_raw):,} rows across all quarters")
print(f"Unique station IDs: {ev_raw['station_id'].nunique():,}")
ev_raw.head(3)


Found 4 EV charging files:
  2025_1T_locations_punts_recarrega_mi.json
  2025_2T_locations_punts_recarrega_mi.json
  2025_3T_locations_punts_recarrega_mi.json
  2025_4T_locations_punts_recarrega_mi.json
  Q1: 137 stations found
  Q2: 138 stations found
  Q3: 136 stations found
  Q4: 133 stations found

Combined EV raw: 544 rows across all quarters
Unique station IDs: 138


,station_id,quarter,network_name,latitude,longitude,address_str,postal_code,locality,access
0,3713,Q1,Endolla Barcelona,41.419664,2.118465,Camí de Vallvidrera al Tibidabo 32,08001,Barcelona,PUBLIC
1,3715,Q1,Endolla Barcelona,41.372802,2.154058,"Av. Reina Maria Cristina, 16",08004,Barcelona,PUBLIC
2,3717,Q1,Endolla Barcelona,41.406310,2.152165,"C/Torrent de l´Olla, 221",08012,Barcelona,PUBLIC


In [ ]:
con.execute("CREATE OR REPLACE TABLE bronze.ev_charging_raw AS SELECT * FROM ev_raw")
count = con.execute("SELECT COUNT(*) FROM bronze.ev_charging_raw").fetchone()[0]
print(f"bronze.ev_charging_raw: {count:,} rows")


bronze.ev_charging_raw: 544 rows


## 6. Vehicle Motorization Index 2025

In [ ]:
# Vehicles motorization index
con.execute("""
    CREATE OR REPLACE TABLE bronze.vehicles_raw AS
    SELECT * FROM read_csv_auto(
        '2025_Parc_vehicles_index_motoritzacio.csv',
        header = true,
        ignore_errors = true
    )
""")
count = con.execute("SELECT COUNT(*) FROM bronze.vehicles_raw").fetchone()[0]
print(f"bronze.vehicles_raw: {count:,} rows")
print("Columns:", [r[0] for r in con.execute("DESCRIBE bronze.vehicles_raw").fetchall()])
print("\nSample Tipus_Vehicle values:")
display(con.execute("SELECT DISTINCT Tipus_Vehicle FROM bronze.vehicles_raw").df())


bronze.vehicles_raw: 6,398 rows
Columns: ['Any', 'Codi_Districte', 'Nom_Districte', 'Codi_Barri', 'Nom_Barri', 'Seccio_Censal', 'Tipus_Vehicle', 'Index_Motoritzacio']

Sample Tipus_Vehicle values:


,Tipus_Vehicle
0,Ciclomotors
1,Furgonetes
2,Motos
3,Resta de vehicles
4,Turismes
5,Camions


We don’t have any analysis or real insight yet, and no answer to our research questions. We just have the raw data in the Bronze layer, being the legal and methodological foundation. Every number in the final report can be traced back to a specific row and table in this layer, which then traces back to a specific downloaded dataset from a specific URL on a specific date.  


## 7. Summarize all the bronze tables

In [ ]:
# Summarize for the data catalogue
for table in ["iris_raw", "accidents_raw", "airquality_raw",
              "airquality_stations_raw", "ev_charging_raw", "vehicles_raw"]:
    print(f"SUMMARIZE bronze.{table}")
    display(con.execute(f"SUMMARIZE bronze.{table}").df())


SUMMARIZE bronze.iris_raw


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,FITXA_ID,VARCHAR,38755903,49201211,292832,<NA>,<NA>,<NA>,<NA>,<NA>,287304,0.00
1,TIPUS,VARCHAR,AGRAIMENT,SUGGESTION,13,<NA>,<NA>,<NA>,<NA>,<NA>,287304,0.00
2,AREA,VARCHAR,Agraïments,Urbanisme,30,<NA>,<NA>,<NA>,<NA>,<NA>,287304,0.00
3,ELEMENT,VARCHAR,A peu o en bicicleta,Òrgans participació districte Sarrià-Sant Gervasi,417,<NA>,<NA>,<NA>,<NA>,<NA>,287304,0.00
4,DETALL,VARCHAR,2a tinència d'alcaldia,eComunicacions(subscripcions i butlletins),1098,<NA>,<NA>,<NA>,<NA>,<NA>,287304,0.00
5,DIA_DATA_ALTA,VARCHAR,01,31,29,<NA>,<NA>,<NA>,<NA>,<NA>,287304,0.00
6,MES_DATA_ALTA,VARCHAR,01,12,12,<NA>,<NA>,<NA>,<NA>,<NA>,287304,0.00
7,ANY_DATA_ALTA,VARCHAR,2023,2025,3,<NA>,<NA>,<NA>,<NA>,<NA>,287304,0.00
8,DIA_DATA_TANCAMENT,VARCHAR,01,31,29,<NA>,<NA>,<NA>,<NA>,<NA>,287304,0.00
9,MES_DATA_TANCAMENT,VARCHAR,01,12,12,<NA>,<NA>,<NA>,<NA>,<NA>,287304,0.00


SUMMARIZE bronze.accidents_raw


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,Número_expedient,VARCHAR,2025S000001,2025S007742,9541,NaN,NaN,NaN,NaN,NaN,8072,0.00
1,Codi_districte,BIGINT,1,10,11,5.090312190287413,3.014505928583992,2,5,8,8072,0.00
2,Nom_districte,VARCHAR,Ciutat Vella,Sarrià-Sant Gervasi,11,NaN,NaN,NaN,NaN,NaN,8072,0.00
3,Codi_barri,BIGINT,1,73,70,28.829410307234888,22.36110669892096,9,22,46,8072,0.00
4,Nom_barri,VARCHAR,Baró de Viver,les Tres Torres,71,NaN,NaN,NaN,NaN,NaN,8072,0.00
5,Codi_carrer,VARCHAR,000180,701920,820,NaN,NaN,NaN,NaN,NaN,8072,0.00
6,Nom_carrer,VARCHAR,A Zona Franca,Àvila,1023,NaN,NaN,NaN,NaN,NaN,8072,0.00
7,Num_postal,VARCHAR,0,S9,1466,NaN,NaN,NaN,NaN,NaN,8072,40.56
8,Descripcio_dia_setmana,VARCHAR,Dijous,Divendres,7,NaN,NaN,NaN,NaN,NaN,8072,0.00
9,Nk_Any,BIGINT,2025,2025,1,2025.0,0.0,2025,2025,2025,8072,0.00


SUMMARIZE bronze.airquality_raw


...

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,Estacio,BIGINT,4,58,7,47.78181818181818,13.784976939610354,43,54,57,55,0.0
1,nom_cabina,VARCHAR,Barcelona - Ciutadella,Barcelona - Vall Hebron,7,NaN,NaN,NaN,NaN,NaN,55,0.0
2,codi_dtes,VARCHAR,I2,IZ,7,NaN,NaN,NaN,NaN,NaN,55,0.0
3,zqa,BIGINT,1,1,1,1.0,0.0,1,1,1,55,0.0
4,codi_eoi,BIGINT,8019004,8019058,7,8019047.781818182,13.784976939808638,8019043,8019054,8019057,55,0.0
5,Longitud,DOUBLE,2.1151,2.2045,9,2.1439309090909084,0.027021821584616367,2.1151,2.148,2.1538,55,0.0
6,Latitud,DOUBLE,41.3788,41.4261,9,41.39767000000002,0.015657778211431713,41.3864,41.3875,41.4039,55,0.0
7,ubicacio,VARCHAR,Av. Roma - c/ Comte Urgell,c/ John Maynard Keynes - c/ de Jordi Girona,7,NaN,NaN,NaN,NaN,NaN,55,0.0
8,Codi_districte,BIGINT,1,10,7,5.090909090909091,2.066243032501325,4,5,6,55,0.0
9,Nom_districte,VARCHAR,Ciutat Vella,Sarria-Sant Gervasi,9,NaN,NaN,NaN,NaN,NaN,55,0.0


SUMMARIZE bronze.ev_charging_raw


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,station_id,VARCHAR,3713,4767,141,NaN,NaN,NaN,NaN,NaN,544,0.0
1,quarter,VARCHAR,Q1,Q4,4,NaN,NaN,NaN,NaN,NaN,544,0.0
2,network_name,VARCHAR,Endolla Barcelona,Endolla Barcelona,1,NaN,NaN,NaN,NaN,NaN,544,0.0
3,latitude,DOUBLE,41.329694,41.459745,127,41.40088159007356,0.021341221045304736,41.3871175510204,41.398780875,41.41137221428571,544,0.0
4,longitude,DOUBLE,2.107238,2.221153,115,2.163705261029411,0.0277014665102662,2.141828071428572,2.1638395,2.185710326530612,544,0.0
5,address_str,VARCHAR,Aparcament Mercabarna,"carrer Puiggarí, 37",135,NaN,NaN,NaN,NaN,NaN,544,0.0
6,postal_code,VARCHAR,08001,08908,37,NaN,NaN,NaN,NaN,NaN,544,0.0
7,locality,VARCHAR,Barcelona,"Hospitalet de Llobregat, L'",2,NaN,NaN,NaN,NaN,NaN,544,0.0
8,access,VARCHAR,PUBLIC,PUBLIC,1,NaN,NaN,NaN,NaN,NaN,544,0.0


SUMMARIZE bronze.vehicles_raw


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,Any,BIGINT,2025,2025,1,2025.0,0.0,2025,2025,2025,6398,0.0
1,Codi_Districte,VARCHAR,01,10,11,NaN,NaN,NaN,NaN,NaN,6398,0.0
2,Nom_Districte,VARCHAR,Ciutat Vella,Sarrià-St. Gervasi,11,NaN,NaN,NaN,NaN,NaN,6398,0.0
3,Codi_Barri,VARCHAR,01,73,76,NaN,NaN,NaN,NaN,NaN,6398,0.0
4,Nom_Barri,VARCHAR,Baró de Viver,les Tres Torres,74,NaN,NaN,NaN,NaN,NaN,6398,0.0
5,Seccio_Censal,VARCHAR,001,237,177,NaN,NaN,NaN,NaN,NaN,6398,0.0
6,Tipus_Vehicle,VARCHAR,Camions,Turismes,6,NaN,NaN,NaN,NaN,NaN,6398,0.0
7,Index_Motoritzacio,VARCHAR,0.56,NA,980,NaN,NaN,NaN,NaN,NaN,6398,0.0


In [ ]:
# Final bronze row count summary
print("\n=== BRONZE LAYER COMPLETE ===")
tables = ["iris_raw","accidents_raw","airquality_raw",
          "airquality_stations_raw","ev_charging_raw","vehicles_raw"]
for t in tables:
    n = con.execute(f"SELECT COUNT(*) FROM bronze.{t}").fetchone()[0]
    print(f"  bronze.{t:35s}: {n:>8,} rows")
print("\nDatabase saved to: barcelona_urban.db")
print("Next step: run 02_silver.ipynb")



=== BRONZE LAYER COMPLETE ===
  bronze.iris_raw                           :  287,304 rows
  bronze.accidents_raw                      :    8,072 rows
  bronze.airquality_raw                     :   22,960 rows
  bronze.airquality_stations_raw            :       55 rows
  bronze.ev_charging_raw                    :      544 rows
  bronze.vehicles_raw                       :    6,398 rows

Database saved to: barcelona_urban.db
Next step: run 02_silver.ipynb


In [ ]:
# Closing the connection before opening the silver layer, bc DuckDB only allows one process to write at a time.
con.close()
print("Connection closed")


Connection closed


: 